# PCB 螺丝孔检测 · YOLOv8n (M2 / MPS)

视觉引导机械臂锁付场景 Demo：训练 → 验证 → TFLite INT8 导出 → 单图推理 → Gradio 界面。

**使用方式：** 从上到下依次运行各单元格；只需改「配置」单元格中的路径与超参。

## 0. 环境与依赖

首次运行执行下面单元格（已安装可跳过）。

In [ ]:
%pip install -q -r requirements.txt

In [8]:
from pathlib import Path
import os

# 确保工作目录在项目根目录
ROOT = Path.cwd()
if not (ROOT / "dataset.yaml").exists():
    ROOT = Path("~/Projects/screw-hole-detector-yolov8").expanduser()
os.chdir(ROOT)
print("Project root:", ROOT.resolve())

Project root: /Users/kcj/projects/screw-hole-detector-yolov8


## 1. 配置（按需修改）

In [10]:
CFG = {
    "data": "dataset.yaml",
    "base_model": "yolov8n.pt",
    "device": "mps",          # M2 用 mps；不可用改为 cpu
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,              # 16GB 内存不够可改为 8 或 4
    "project": "runs/detect",
    "name": "screw_hole",
    "patience": 50,
    "workers": 4,
    "resume": False,
    "conf": 0.35,
}


def find_best_pt(root: Path, cfg: dict) -> Path:
    """Resolve best.pt (handles nested runs/detect/runs/detect/...)."""
    primary = root / cfg["project"] / cfg["name"] / "weights" / "best.pt"
    if primary.exists():
        return primary.resolve()
    matches = [
        p
        for p in root.glob("**/weights/best.pt")
        if cfg["name"] in str(p) and "weights" in str(p)
    ]
    if matches:
        return max(matches, key=lambda p: p.stat().st_mtime).resolve()
    return primary.resolve()


# 训练/导出统一用 ROOT 下的绝对路径（需先运行「项目根目录」单元格）
BEST_PT = find_best_pt(ROOT, CFG)
TRAIN_PROJECT = str((ROOT / CFG["project"]).resolve())

print("BEST_PT:", BEST_PT, "| exists:", BEST_PT.exists())
print("TRAIN_PROJECT:", TRAIN_PROJECT)
CFG

BEST_PT: /Users/kcj/projects/screw-hole-detector-yolov8/runs/detect/screw_hole/weights/best.pt | exists: True
TRAIN_PROJECT: /Users/kcj/projects/screw-hole-detector-yolov8/runs/detect


{'data': 'dataset.yaml',
 'base_model': 'yolov8n.pt',
 'device': 'mps',
 'epochs': 100,
 'imgsz': 640,
 'batch': 16,
 'project': 'runs/detect',
 'name': 'screw_hole',
 'patience': 50,
 'workers': 4,
 'resume': False,
 'conf': 0.35}

## 2. 检查数据集

目录结构见 README：`data/screw_holes/images/{train,val}` 与对应 `labels/`。

In [3]:
import yaml

with open(CFG["data"]) as f:
    ds = yaml.safe_load(f)

data_root = ROOT / ds["path"]
train_img = data_root / ds["train"]
val_img = data_root / ds["val"]

def count_images(p: Path) -> int:
    if not p.exists():
        return 0
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sum(1 for x in p.rglob("*") if x.suffix.lower() in exts)

print("Dataset root:", data_root.resolve())
print("Train images:", count_images(train_img), "|", train_img)
print("Val images:  ", count_images(val_img), "|", val_img)
print("Classes:", ds.get("names"))

if count_images(train_img) == 0:
    print("\n⚠️ 训练集为空，请先放入标注数据再训练。")

Dataset root: /Users/kcj/projects/screw-hole-detector-yolov8/data/screw_holes
Train images: 80 | /Users/kcj/projects/screw-hole-detector-yolov8/data/screw_holes/images/train
Val images:   20 | /Users/kcj/projects/screw-hole-detector-yolov8/data/screw_holes/images/val
Classes: {0: 'screw_hole'}


## 3. 训练 YOLOv8n（MPS + 自动保存 best）

In [4]:
from ultralytics import YOLO

assert Path(CFG["data"]).exists(), f"缺少 {CFG['data']}"
os.chdir(ROOT)  # 必须在项目根训练，否则会生成 runs/detect/runs/detect/...

model = YOLO(CFG["base_model"])
results = model.train(
    data=str(ROOT / CFG["data"]),
    epochs=CFG["epochs"],
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    device=CFG["device"],
    project=TRAIN_PROJECT,
    name=CFG["name"],
    patience=CFG["patience"],
    workers=CFG["workers"],
    resume=CFG["resume"],
    save=True,
    plots=True,
    val=True,
    pretrained=True,
)

BEST_PT = find_best_pt(ROOT, CFG)
print("Best weights:", BEST_PT, "| exists:", BEST_PT.exists())
print("Results:", getattr(results, "results_dict", results))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/kcj/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.60 🚀 Python-3.11.5 torch-2.12.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, in

## 4. TensorBoard（训练曲线）

在 Jupyter 内嵌显示；若无效可在终端运行：`tensorboard --logdir runs/detect`

In [5]:
%load_ext tensorboard
%tensorboard --logdir runs/detect --port 6006

## 5. 导出 TFLite INT8（边缘部署）

In [11]:
import os
import time

# 少刷屏；Mac 无 CUDA 时 "0 eligible GPUs" 是正常提示
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

EXPORT_INT8 = False  # True=INT8 更小更慢(5–20min+)；False=float32 先快速出文件
EXPORT_DEVICE = "cpu"  # TFLite 导出建议 cpu，不要用 mps
EXPORT_IMGSZ = 640     # 可先改 320 加快

BEST_PT = find_best_pt(ROOT, CFG)
if not BEST_PT.exists():
    raise FileNotFoundError(f"找不到 best.pt: {BEST_PT}")

print("开始导出… Mac 上 TensorFlow 日志可能停住很久，请看活动监视器 CPU 是否还在跑。")
print(f"  int8={EXPORT_INT8}  device={EXPORT_DEVICE}  imgsz={EXPORT_IMGSZ}")
t0 = time.time()

export_model = YOLO(str(BEST_PT))
tflite_path = export_model.export(
    format="tflite",
    imgsz=EXPORT_IMGSZ,
    device=EXPORT_DEVICE,
    int8=EXPORT_INT8,
    data=str(ROOT / CFG["data"]) if EXPORT_INT8 else None,
)
TFLITE_PATH = Path(tflite_path)
print(f"完成，用时 {time.time()-t0:.0f}s")
print("TFLite:", TFLITE_PATH.resolve())

Ultralytics 8.4.60 🚀 Python-3.11.5 torch-2.12.0 MPS (Apple M2)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/Users/kcj/projects/screw-hole-detector-yolov8/runs/detect/screw_hole/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
TensorFlow SavedModel: collecting INT8 calibration images from 'data=dataset.yaml'
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 354.2±115.9 MB/s, size: 107.1 KB)
val: Scanning /Users/kcj/projects/screw-hole-detector-yolov8/data/screw_holes/labels/val.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 2.0Mit/s 0.0s
WARNING ⚠️ TensorFlow SavedModel: >300 images recommended for INT8 calibration, found 20 images.
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
  Obtaining dependency information for onnx<2.0.0,>=1.12.0 from https://files.pythonhosted.or

fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


TensorFlow SavedModel: export success ✅ 223.1s, saved as '/Users/kcj/projects/screw-hole-detector-yolov8/runs/detect/screw_hole/weights/best_saved_model' (39.0 MB)

TensorFlow Lite: starting export with tensorflow 2.21.0...
TensorFlow Lite: export success ✅ 0.0s, saved as '/Users/kcj/projects/screw-hole-detector-yolov8/runs/detect/screw_hole/weights/best_saved_model/best_int8.tflite' (3.2 MB)

Export complete (225.5s)
Results saved to /Users/kcj/projects/screw-hole-detector-yolov8/runs/detect/screw_hole/weights/best_saved_model/best_int8.tflite
Predict:         yolo predict task=detect model=/Users/kcj/projects/screw-hole-detector-yolov8/runs/detect/screw_hole/weights/best_saved_model/best_int8.tflite imgsz=640 int8
Validate:        yolo val task=detect model=/Users/kcj/projects/screw-hole-detector-yolov8/runs/detect/screw_hole/weights/best_saved_model/best_int8.tflite imgsz=640 data=dataset.yaml int8 
Visualize:       https://netron.app
TFLite: /Users/kcj/projects/screw-hole-detector-

## 6. 单张图片预测（Notebook 内显示）

修改 `IMAGE_PATH` 为你的 PCB 图片路径。

In [13]:
from IPython.display import Image as IPImage, display
import matplotlib.pyplot as plt

IMAGE_PATH = "data/screw_holes/images/val/example.jpg"  # ← 改成你的图片
USE_TFLITE = False  # True 则用上面导出的 TFLite

weights = TFLITE_PATH if USE_TFLITE else BEST_PT
if not Path(weights).exists():
    raise FileNotFoundError(weights)

pred_model = YOLO(str(weights))
device = "cpu" if USE_TFLITE or str(weights).endswith(".tflite") else CFG["device"]

pred_results = pred_model.predict(
    source=IMAGE_PATH,
    conf=CFG["conf"],
    imgsz=CFG["imgsz"],
    device=device,
    save=False,
    verbose=False,
)
r = pred_results[0]
annotated = r.plot()[:, :, ::-1]

plt.figure(figsize=(10, 8))
plt.imshow(annotated)
plt.axis("off")
plt.title("Screw-hole detection")
plt.show()

boxes = r.boxes
n = 0 if boxes is None else len(boxes)
print(f"Detected: {n} screw hole(s)")
if boxes is not None:
    for i, box in enumerate(boxes):
        xyxy = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        cx, cy = (xyxy[0] + xyxy[2]) / 2, (xyxy[1] + xyxy[3]) / 2
        print(f"  #{i+1} conf={conf:.3f} bbox={[round(x,1) for x in xyxy]} center=({cx:.1f},{cy:.1f})")

<Figure size 1000x800 with 1 Axes>

Detected: 7 screw hole(s)
  #1 conf=0.982 bbox=[245.0, 434.9, 278.1, 466.9] center=(261.5,450.9)
  #2 conf=0.967 bbox=[195.4, 396.8, 227.1, 429.1] center=(211.3,413.0)
  #3 conf=0.960 bbox=[108.0, 124.0, 140.2, 156.1] center=(124.1,140.0)
  #4 conf=0.958 bbox=[228.8, 415.9, 261.0, 448.1] center=(244.9,432.0)
  #5 conf=0.954 bbox=[299.3, 210.1, 331.5, 242.1] center=(315.4,226.1)
  #6 conf=0.929 bbox=[82.5, 366.8, 107.4, 391.0] center=(95.0,378.9)
  #7 conf=0.907 bbox=[392.5, 339.9, 424.4, 372.2] center=(408.5,356.1)


## 7. Gradio Web Demo（支持 .pt / .tflite）

运行后在 Notebook 内嵌界面；也可访问终端打印的本地 URL。

In [14]:
import tempfile
import gradio as gr
import numpy as np
from PIL import Image

_cache: dict[str, YOLO] = {}

def _load(w: str) -> YOLO:
    p = str(Path(w).expanduser().resolve())
    if p not in _cache:
        _cache[p] = YOLO(p)
    return _cache[p]

def infer(img, backend, wpath, conf, imgsz):
    if img is None:
        return None, "请上传图片", ""
    if isinstance(img, np.ndarray):
        img = Image.fromarray(img)
    try:
        m = _load(wpath)
    except Exception as e:
        return None, str(e), ""
    dev = "cpu" if backend == "TFLite" else CFG["device"]
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as t:
        img.convert("RGB").save(t.name)
        tmp = t.name
    try:
        res = m.predict(source=tmp, conf=conf, imgsz=int(imgsz), device=dev, verbose=False)[0]
    finally:
        Path(tmp).unlink(missing_ok=True)
    out = Image.fromarray(res.plot()[:, :, ::-1])
    boxes = res.boxes
    cnt = 0 if boxes is None else len(boxes)
    lines, rows = [], []
    if boxes is not None:
        for i, b in enumerate(boxes):
            xy = b.xyxy[0].tolist()
            c = float(b.conf[0])
            cx, cy = (xy[0]+xy[2])/2, (xy[1]+xy[3])/2
            lines.append(f"#{i+1} conf={c:.3f} center=({cx:.0f},{cy:.0f})")
            rows.append([i+1, f"{c:.3f}", f"({xy[0]:.0f},{xy[1]:.0f})", f"({xy[2]:.0f},{xy[3]:.0f})", f"({cx:.0f},{cy:.0f})"])
    md = f"**{cnt}** 个螺丝孔\n\n" + "\n".join(lines)
    tbl = ""
    if rows:
        tbl = "| # | conf | 左上 | 右下 | 中心 |\n|---:|---:|---|---|---|\n"
        tbl += "\n".join(f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]} |" for r in rows)
    return out, md, tbl

pt_default = str(BEST_PT)
tflite_default = str(TFLITE_PATH) if 'TFLITE_PATH' in dir() and TFLITE_PATH.exists() else pt_default.replace('.pt', '_int8.tflite')

with gr.Blocks(title="螺丝孔检测", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# PCB 螺丝孔检测 · YOLOv8")
    with gr.Row():
        with gr.Column():
            bk = gr.Radio(["PyTorch (.pt)", "TFLite"], value="PyTorch (.pt)", label="后端")
            wp = gr.Textbox(label="模型路径", value=pt_default)
            cf = gr.Slider(0.05, 0.95, value=CFG["conf"], label="置信度")
            iz = gr.Slider(320, 1280, value=CFG["imgsz"], step=32, label="尺寸")
            inp = gr.Image(type="pil", label="上传 PCB")
            btn = gr.Button("检测", variant="primary")
        with gr.Column():
            outp = gr.Image(type="pil", label="结果")
            summ = gr.Markdown()
            tabl = gr.Markdown()
    bk.change(lambda b: tflite_default if b == "TFLite" else pt_default, bk, wp)
    btn.click(infer, [inp, bk, wp, cf, iz], [outp, summ, tabl])
    inp.change(infer, [inp, bk, wp, cf, iz], [outp, summ, tabl])

demo.launch(server_name="127.0.0.1", server_port=7860, share=False, inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
